In [26]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

In [58]:
model_name = 'MVP-FAS similarity'  # MVP-FAS | DMFC-FAS | CVPR2024-FAS
domain_name = 'figure similarity w=0,5'  # cutout | figure | mannequin | mask | print | screen

path = Path(model_name) / domain_name / 'scores.csv'
path

WindowsPath('MVP-FAS similarity/figure similarity w=0,5/scores.csv')

In [59]:
df = pd.read_csv(path)
df

,step,score,params
0,0,0.293976,"{'idx_choice--0': 1, 'idx_choice--1': 1, 'idx_..."
1,1,0.452948,"{'idx_choice--0': 0, 'scale--left--Morphologic..."
2,2,0.303731,"{'idx_choice--0': 0, 'scale--left--Morphologic..."
3,3,0.432173,"{'idx_choice--0': 0, 'scale--left--Morphologic..."
4,4,0.368329,"{'idx_choice--0': 0, 'scale--left--Morphologic..."
...,...,...,...
495,495,0.518744,"{'idx_choice--0': 1, 'idx_choice--1': 0, 'drop..."
496,496,0.830087,"{'idx_choice--0': 1, 'idx_choice--1': 0, 'drop..."
497,497,0.530302,"{'idx_choice--0': 1, 'idx_choice--1': 0, 'drop..."
498,498,0.848234,"{'idx_choice--0': 1, 'idx_choice--1': 0, 'drop..."


In [60]:
df.iloc[0, 2]

"{'idx_choice--0': 1, 'idx_choice--1': 1, 'idx_choice--2': 7, 'idx_choice--3': 2, 'blur_limit--left--MedianBlurTransform': 4, 'blur_limit--right--MedianBlurTransform': 5, 'idx_choice--4': 1, 'snow_point_range--left--SnowTransform': 0.5586792581651092, 'snow_point_range--right--SnowTransform': 0.8791445226883231, 'brightness_coeff--SnowTransform': 1.3782344584985378, 'idx_choice--5': 2, 'shift_limit--left--ShiftScaleRotateTransform': -0.10195894246030836, 'shift_limit--right--ShiftScaleRotateTransform': 0.0779034346750377, 'scale_limit--left--ShiftScaleRotateTransform': 0.9881458486488404, 'scale_limit--right--ShiftScaleRotateTransform': 0.9994157982914103, 'rotate_limit--left--ShiftScaleRotateTransform': 208.71876353210632, 'rotate_limit--right--ShiftScaleRotateTransform': 246.28435437912012, 'fill--ShiftScaleRotateTransform': 112, 'idx_choice--6': 1, 'scale_range--left--DownscaleTransform': 0.4058709541311646, 'scale_range--right--DownscaleTransform': 0.8282975782054819, 'idx_choice--

In [61]:
def get_set_transforms(params: str | dict) -> set:
    if isinstance(params, str):
        params = eval(params)

    set_transforms = set()
    for key in params.keys():
        if 'idx_choice' in key:
            continue

        name = key.split('--')[-1]
        set_transforms.add(name)
    return set_transforms


get_set_transforms(df.iloc[0, 2])

{'CoarseDropoutTransform',
 'DownscaleTransform',
 'MedianBlurTransform',
 'ShiftScaleRotateTransform',
 'SnowTransform'}

In [62]:
data = defaultdict(dict)
for i in tqdm(range(df.shape[0])):
    step = df.iloc[i, 0]
    score = df.iloc[i, 1]
    set_transforms = get_set_transforms(df.iloc[i, 2])

    data[step]['score'] = score
    data[step]['set_transforms'] = set_transforms

data[0]

100%|████████████████████████████████████████████| 500/500 [00:00<00:00, 2073.09it/s]


{'score': 0.2939763744186729,
 'set_transforms': {'CoarseDropoutTransform',
  'DownscaleTransform',
  'MedianBlurTransform',
  'ShiftScaleRotateTransform',
  'SnowTransform'}}

In [63]:
def get_importance(data: dict) -> pd.DataFrame:
    dict_scores = defaultdict(float)  # transform: score
    for info in data.values():
        for name in info['set_transforms']:
            dict_scores[name] += info['score']

    
    df = pd.DataFrame({'name': [i for i in dict_scores.keys()], 'score': [i for i in dict_scores.values()]})
    sum_score = df.score.sum()
    df['score'] /= sum_score
    df = df.sort_values(by='score', ascending=False)
    return df

In [64]:
df_importance = get_importance(data)
df_importance

,name,score
6,GridDropoutTransform,0.205560
11,PixelDropoutTransform,0.165528
19,ZoomBlurTransform,0.155111
26,HSVTransform,0.150409
14,GaussianBlurTransform,0.112363
0,MedianBlurTransform,0.036457
25,BlurTransform,0.027585
18,RGBShiftTransform,0.023008
23,BrightnessContrastTransform,0.021683
2,DownscaleTransform,0.013545


In [65]:
from typing import Iterable
import matplotlib.pyplot as plt


def save_importance_barh(
    names: Iterable,
    values: Iterable,
    path: Path,
    figsize: tuple = (13, 5),
    title: str | None = None,
    xlabel: str | None = None,
    ylabel: str | None = None,
    color: str = '#2196F3',
) -> None:
    """"""
    plt.figure(figsize=figsize)
    bars = plt.barh(
        names,
        values,
        color=color,
    )

    for bar, val in zip(bars, values):
        width = bar.get_width()
        plt.text(width + max(values) * 0.01, bar.get_y() + bar.get_height() / 2, 
                f'{val:.4f}', ha='left', va='center', fontsize=8)

    if xlabel:
        plt.xlabel(xlabel)
    if ylabel:
        plt.ylabel(ylabel)
    if title:
        plt.title(title)

    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

In [66]:
df_importance = df_importance.head(5)


# save_importance_barh(
#     names=df_importance.name.values,
#     values=df_importance.score.values,
#     path=Path(model_name) / domain_name / 'param_importance_plot_v2.png',
#     title=f'Важность transforms в {model_name} {domain_name}',
#     xlabel='score',
#     ylabel='transform'
# )

save_importance_barh(
    names=df_importance.name.values,
    values=df_importance.score.values,
    path=Path(model_name) / domain_name / 'param_importance_plot_v3.png',
    title=f'Важность transforms в {model_name} {domain_name}',
    xlabel='score',
    ylabel='transform'
)

In [17]:
mapping = {
    'ShearTransform': 'Affine',
    'PerspectiveTransform': 'Perspective',
    'GridDistortionTransform': 'GridDistortion',
    'OpticalDistortionTransform': 'OpticalDistortion',
    'ShiftScaleRotateTransform': 'ShiftScaleRotate',
    'BrightnessContrastTransform': 'RandomBrightnessContrast',
    'HSVTransform': 'HueSaturationValue',
    'RGBShiftTransform': 'RGBShift',
    'GammaTransform': 'RandomGamma',
    'CLAHETransform': 'CLAHE',
    'SolarizeTransform': 'Solarize',
    'PosterizeTransform': 'Posterize',
    'EqualizeTransform': 'Equalize',
    'InvertTransform': 'InvertImg',
    'ToGrayTransform': 'ToGray',
    'ChannelShuffleTransform': 'ChannelShuffle',
    'ToSepiaTransform': 'ToSepia',
    'BlurTransform': 'Blur',
    'GaussianBlurTransform': 'GaussianBlur',
    'MedianBlurTransform': 'MedianBlur',
    'MotionBlurTransform': 'MotionBlur',
    'SharpenTransform': 'Sharpen',
    'EmbossTransform': 'Emboss',
    'GaussNoiseTransform': 'GaussNoise',
    'MultiplicativeNoiseTransform': 'MultiplicativeNoise',
    'ISONoiseTransform': 'ISONoise',
    'CoarseDropoutTransform': 'CoarseDropout',
    'GridDropoutTransform': 'GridDropout',
    'CompressionTransform': 'ImageCompression',
    'DownscaleTransform': 'Downscale',
    'PixelDropoutTransform': 'PixelDropout',
    'RainTransform': 'RandomRain',
    'SnowTransform': 'RandomSnow',
    'FogTransform': 'RandomFog',
    'ShadowTransform': 'RandomShadow',
    'SunFlareTransform': 'RandomSunFlare',
    'RainbowTransform': 'RandomToneCurve',
    'SpatterTransform': 'Spatter',
    'ChromaticAberrationTransform': 'ChromaticAberration',
    'DefocusTransform': 'Defocus',
    'ZoomBlurTransform': 'ZoomBlur',
    'MorphologicalTransform': 'Morphological',
    'ShotNoiseTransform': 'ShotNoise',
}

In [25]:
# list_domain_rus = ['вырезка', 'фигура', 'манекен', 'маска', 'распечатка', 'экран']
list_domain_rus = ['w=0,5', 'w=0,3', 'w=0,1']


# for model_name in ('CVPR2024-FAS', 'MVP-FAS', 'DMFC-FAS'):
for model_name in ('MVP-FAS similarity',):
    k = 0
    # for domain_name in ('cutout', 'figure', 'mannequin', 'mask', 'print', 'screen'):
    for domain_name in ('figure similarity w=0,5', 'figure similarity w=0,3', 'figure similarity w=0,1'):
        print(model_name, domain_name)
        path = Path(model_name) / domain_name / 'scores.csv'
        df = pd.read_csv(path)

        data = defaultdict(dict)
        for i in tqdm(range(df.shape[0])):
            step = df.iloc[i, 0]
            score = df.iloc[i, 1]
            set_transforms = get_set_transforms(df.iloc[i, 2])
        
            data[step]['score'] = score
            data[step]['set_transforms'] = set_transforms

        df_importance = get_importance(data)
        df_importance = df_importance.head(10)
        df_importance.name = df_importance.name.map(lambda x: mapping[x])

        save_importance_barh(
            names=df_importance.name.values,
            values=df_importance.score.values,
            path=Path(model_name) / domain_name / 'param_importance_plot_v2.png',
            title=f'Важность transforms в {model_name} `{list_domain_rus[k]}`',
            xlabel='score',
            ylabel='transform',
        )
        k += 1

MVP-FAS similarity figure similarity w=0,5


100%|████████████████████████████████████████████| 500/500 [00:00<00:00, 1340.97it/s]


MVP-FAS similarity figure similarity w=0,3


100%|████████████████████████████████████████████| 500/500 [00:00<00:00, 1594.73it/s]


MVP-FAS similarity figure similarity w=0,1


100%|████████████████████████████████████████████| 500/500 [00:00<00:00, 1734.35it/s]
